In [ ]:
# Generated deterministic BigAlpha submission.
import re

try:
    import dai
except ModuleNotFoundError as exc:
    if exc.name != "dai":
        raise
    # BigQuant AIStudio injects ``dai`` directly.  The official local SDK
    # exposes the same client as ``bigquant.dai``.  Keeping this import boundary
    # inside the standalone artifact lets the exact same source run in both
    # environments without changing factor semantics.
    from bigquant import dai
import numpy as np
import pandas as pd

_BAR_MAP = {'adjust_factor': 'adjust_factor', 'amount': 'amount', 'close': 'close', 'date': 'date', 'datetime': 'date', 'high': 'high', 'instrument': 'instrument', 'low': 'low', 'open': 'open', 'volume': 'volume'}
_MINUTE_VOLATILITY_FIELDS = ['minute_downside_semivariance', 'minute_realized_volatility', 'minute_upside_semivariance']
_EXPECTED_SESSION_MINUTES = np.concatenate(
    (
        np.arange(9 * 60 + 31, 11 * 60 + 31, dtype=np.int16),
        np.arange(13 * 60 + 1, 15 * 60 + 1, dtype=np.int16),
    )
)
_COUNTER_SEMANTICS = {'amount': 'per_bar', 'volume': 'per_bar'}
_UNIVERSE_MAP = {'date': 'date', 'instrument': 'instrument'}
_UNIVERSE_TABLE = 'bigalpha_2026_instruments'
_FIELD_NAMES = ['close', 'low']
_SCORE_DIRECTION = -1
_LOOKBACK = 50
_SAFE_DIVIDE_MIN_ABS = 1e-12
_CHUNK_DAYS = 7
_IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_.]*$")


def _table(value):
    if not isinstance(value, str) or _IDENTIFIER.fullmatch(value) is None:
        raise ValueError("invalid datasource table identifier")
    return value


def _query(table, columns, start_date, end_date):
    table = _table(table)
    selected = sorted(set(columns))
    if any(_IDENTIFIER.fullmatch(column) is None for column in selected):
        raise ValueError("invalid source column identifier")
    sql = "SELECT " + ", ".join(selected) + " FROM " + table
    result = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        full_db_scan=True,
    )
    frame = result.df() if callable(getattr(result, "df", None)) else result
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("dai.query must return a pandas DataFrame or .df() result")
    missing = sorted(set(selected) - set(frame.columns))
    if missing:
        raise ValueError("query result is missing columns: " + repr(missing))
    return frame.copy()


def _date_chunks(start_date, end_date):
    cursor = pd.Timestamp(start_date).normalize()
    end = pd.Timestamp(end_date).normalize()
    while cursor <= end:
        chunk_end = min(cursor + pd.Timedelta(days=_CHUNK_DAYS - 1), end)
        yield cursor.strftime("%Y-%m-%d"), chunk_end.strftime("%Y-%m-%d")
        cursor = chunk_end + pd.Timedelta(days=1)


def _daily_bars(raw):
    frame = pd.DataFrame(
        {canonical: raw[actual] for canonical, actual in _BAR_MAP.items()}
    )
    frame["instrument"] = frame["instrument"].astype(str).str.strip()
    frame["datetime"] = pd.to_datetime(frame["datetime"], errors="coerce")
    if frame["instrument"].eq("").any() or frame["datetime"].isna().any():
        raise ValueError("bar keys contain invalid values")
    if isinstance(frame["datetime"].dtype, pd.DatetimeTZDtype):
        frame["datetime"] = (
            frame["datetime"].dt.tz_convert("Asia/Shanghai").dt.tz_localize(None)
        )
    frame["date"] = frame["datetime"].dt.normalize()
    numeric = ["open", "high", "low", "close", "volume", "amount"]
    for optional in ("trade_count", "adjust_factor"):
        if optional in frame:
            numeric.append(optional)
    for column in numeric:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    if not np.isfinite(frame[numeric].to_numpy(dtype=float)).all():
        raise ValueError("bar values must be finite numeric data")
    if frame.duplicated(["date", "instrument", "datetime"]).any():
        raise ValueError("duplicate minute bars")
    frame = frame.sort_values(
        ["instrument", "date", "datetime"], kind="mergesort", ignore_index=True
    )
    keys = ["instrument", "date"]
    grouped = frame.groupby(keys, sort=False, observed=True)
    if "adjust_factor" in frame:
        if (frame["adjust_factor"] <= 0.0).any():
            raise ValueError("adjust_factor must be positive")
        if (grouped["adjust_factor"].nunique(dropna=False) != 1).any():
            raise ValueError("adjust_factor changes within a session")
        for column in ("open", "high", "low", "close"):
            frame[column] = frame[column] * frame["adjust_factor"]
    counters = ["volume", "amount"]
    if "trade_count" in frame:
        counters.append("trade_count")
    if set(_COUNTER_SEMANTICS) != set(counters):
        raise ValueError("counter semantics do not match mapped counters")
    grouped = frame.groupby(keys, sort=False, observed=True)
    for column in counters:
        mode = _COUNTER_SEMANTICS[column]
        if mode == "session_cumulative":
            previous = grouped[column].shift(1)
            component = frame[column].where(
                previous.isna(), frame[column] - previous
            )
            error = "cumulative counter decreases within a session"
        elif mode == "per_bar":
            component = frame[column]
            error = "per-bar counter must be non-negative"
        else:
            raise ValueError("unsupported counter semantics")
        if (component < 0.0).any():
            raise ValueError(error)
        frame["__daily_component_" + column] = component
    aggregations = {
        "open": ("open", "first"),
        "high": ("high", "max"),
        "low": ("low", "min"),
        "close": ("close", "last"),
        "volume": ("__daily_component_volume", "sum"),
        "amount": ("__daily_component_amount", "sum"),
    }
    if "trade_count" in frame:
        aggregations["trade_count"] = ("__daily_component_trade_count", "sum")
    if "adjust_factor" in frame:
        aggregations["adjust_factor"] = ("adjust_factor", "first")
    daily = frame.groupby(keys, sort=True, observed=True).agg(**aggregations).reset_index()
    daily["vwap"] = np.divide(
        daily["amount"],
        daily["volume"],
        out=np.full(len(daily), np.nan),
        where=daily["volume"].to_numpy() != 0.0,
    )
    if "adjust_factor" in daily:
        daily["vwap"] = daily["vwap"] * daily["adjust_factor"]
    requested_minute_fields = sorted(
        set(_FIELD_NAMES) & set(_MINUTE_VOLATILITY_FIELDS)
    )
    if requested_minute_fields:
        daily = daily.merge(
            _minute_volatility_atoms(frame, requested_minute_fields),
            on=["instrument", "date"],
            how="left",
            validate="one_to_one",
            sort=False,
        )
    return daily


def _minute_volatility_atoms(frame, fields):
    keys = ["instrument", "date"]
    grouped = frame.groupby(keys, sort=False, observed=True)
    positions = grouped.cumcount()
    counts = grouped["datetime"].transform("size")
    position_values = positions.to_numpy(dtype=np.int64, copy=False)
    valid_positions = position_values < len(_EXPECTED_SESSION_MINUTES)
    expected_minutes = np.full(len(frame), -1, dtype=np.int16)
    expected_minutes[valid_positions] = _EXPECTED_SESSION_MINUTES[
        position_values[valid_positions]
    ]
    timestamps = frame["datetime"].dt
    observed_minutes = (
        timestamps.hour.to_numpy(dtype=np.int16, copy=False) * 60
        + timestamps.minute.to_numpy(dtype=np.int16, copy=False)
    )
    exact_grid = (
        valid_positions
        & (observed_minutes == expected_minutes)
        & timestamps.second.eq(0).to_numpy(dtype=bool, copy=False)
        & timestamps.microsecond.eq(0).to_numpy(dtype=bool, copy=False)
    )
    positive_prices = frame[["open", "high", "low", "close"]].gt(0.0).all(axis=1)
    ohlc_consistent = (
        frame["high"].ge(frame[["open", "close", "low"]].max(axis=1))
        & frame["low"].le(frame[["open", "close", "high"]].min(axis=1))
    )
    frame["__minute_row_valid"] = (
        exact_grid
        & positive_prices.to_numpy(dtype=bool, copy=False)
        & ohlc_consistent.to_numpy(dtype=bool, copy=False)
    )
    grouped = frame.groupby(keys, sort=False, observed=True)
    exact_session = grouped["__minute_row_valid"].transform("all") & counts.eq(
        len(_EXPECTED_SESSION_MINUTES)
    )
    volume_total = grouped["__daily_component_volume"].transform("sum")
    amount_total = grouped["__daily_component_amount"].transform("sum")
    session_usable = exact_session & ((volume_total > 0.0) | (amount_total > 0.0))
    previous = grouped["close"].shift(1)
    previous = previous.where(positions.ne(0), frame["open"])
    valid_return = frame["close"].gt(0.0) & previous.gt(0.0)
    returns = np.full(len(frame), np.nan, dtype=float)
    valid_return_values = valid_return.to_numpy(dtype=bool, copy=False)
    returns[valid_return_values] = np.log(
        frame.loc[valid_return, "close"].to_numpy(dtype=float, copy=False)
        / previous.loc[valid_return].to_numpy(dtype=float, copy=False)
    )
    frame["__minute_squared_return"] = np.square(returns)
    frame["__minute_downside_squared_return"] = np.square(
        np.minimum(returns, 0.0)
    )
    frame["__minute_upside_squared_return"] = np.square(
        np.maximum(returns, 0.0)
    )
    frame["__minute_session_usable"] = session_usable
    atoms = (
        frame.groupby(keys, sort=True, observed=True)
        .agg(
            __realized_variance=("__minute_squared_return", "sum"),
            minute_downside_semivariance=(
                "__minute_downside_squared_return",
                "sum",
            ),
            minute_upside_semivariance=(
                "__minute_upside_squared_return",
                "sum",
            ),
            __session_usable=("__minute_session_usable", "all"),
        )
        .reset_index()
    )
    atoms["minute_realized_volatility"] = np.sqrt(atoms["__realized_variance"])
    unusable = ~atoms["__session_usable"].astype(bool)
    atoms.loc[unusable, _MINUTE_VOLATILITY_FIELDS] = np.nan
    return atoms[[*keys, *fields]]


def _align_universe(daily, raw_membership):
    members = pd.DataFrame(
        {
            "date": pd.to_datetime(
                raw_membership[_UNIVERSE_MAP["date"]], errors="coerce"
            ).dt.normalize(),
            "instrument": raw_membership[_UNIVERSE_MAP["instrument"]]
            .astype(str)
            .str.strip(),
        }
    )
    flag = _UNIVERSE_MAP.get("is_member")
    if flag is not None:
        if raw_membership[flag].isna().any():
            raise ValueError("membership flag contains null values")
        members = members.loc[raw_membership[flag].astype(bool)].copy()
    if members.isna().any().any() or members["instrument"].eq("").any():
        raise ValueError("membership keys contain invalid values")
    if members.duplicated(["date", "instrument"]).any():
        raise ValueError("duplicate membership keys")
    return members.merge(
        daily,
        on=["date", "instrument"],
        how="left",
        validate="one_to_one",
        sort=False,
    ).sort_values(["date", "instrument"], kind="mergesort", ignore_index=True)


def _align_pair(x, y):
    x, y = x.align(y, join="inner", axis=0)
    return x, y.reindex(columns=x.columns)


def _broadcast_numeric_operands(*operands):
    frames = [operand for operand in operands if isinstance(operand, pd.DataFrame)]
    reference = None
    if frames:
        first = frames[0]
        common_index = first.index
        for frame in frames[1:]:
            common_index = common_index.intersection(frame.index, sort=False)
        reference = first.reindex(index=common_index)
        values = tuple(
            (
                operand.reindex(
                    index=reference.index, columns=reference.columns
                ).to_numpy(dtype=float, copy=False)
                if isinstance(operand, pd.DataFrame)
                else np.asarray(operand, dtype=float)
            )
            for operand in operands
        )
        return reference, values
    return None, tuple(np.asarray(operand, dtype=float) for operand in operands)


def _broadcast_result(reference, values):
    result = np.asarray(values, dtype=float)
    if reference is None:
        return float(result) if result.ndim == 0 else result
    if result.shape != reference.shape:
        result = np.broadcast_to(result, reference.shape)
    return pd.DataFrame(result, index=reference.index, columns=reference.columns)


def _add(x, y): return x + y
def _sub(x, y): return x - y
def _mul(x, y): return x * y
def _neg(x): return -x
def _abs(x): return x.abs()
def _scale(x, value): return x * float(value)
def _add_const(x, value): return x + float(value)
def _sign(x): return np.sign(x)
def _positive_part(x): return x.clip(lower=0.0)
def _negative_part(x): return (-x).clip(lower=0.0)
def _log1p_abs(x): return np.log1p(x.abs())
def _log_positive(x):
    reference, (values,) = _broadcast_numeric_operands(x)
    with np.errstate(divide="ignore", invalid="ignore"):
        logged = np.log(np.where(values > 0.0, values, np.nan))
    return _broadcast_result(reference, logged)

def _sqrt_abs(x): return np.sqrt(x.abs())
def _signed_power(x, power): return np.sign(x) * np.power(x.abs(), float(power))
def _signed_power_dynamic(x, exponent):
    reference, (base_values, exponent_values) = _broadcast_numeric_operands(x, exponent)
    valid = np.isfinite(base_values) & np.isfinite(exponent_values)
    undefined_zero = (base_values == 0.0) & (exponent_values < 0.0)
    safe_base = np.where(valid & ~undefined_zero, np.abs(base_values), 1.0)
    with np.errstate(over="ignore", invalid="ignore"):
        powered = np.sign(base_values) * np.power(safe_base, exponent_values)
    usable = valid & ~undefined_zero & np.isfinite(powered)
    return _broadcast_result(reference, np.where(usable, powered, np.nan))

def _clip(x, lower, upper): return x.clip(lower=float(lower), upper=float(upper))


def _safe_div(x, y):
    reference, (numerator, denominator) = _broadcast_numeric_operands(x, y)
    denominator = np.where(
        np.abs(denominator) > _SAFE_DIVIDE_MIN_ABS,
        denominator,
        np.nan,
    )
    return _broadcast_result(reference, numerator / denominator)


def _floor_abs(x, eps):
    eps = abs(float(eps))
    reference, (values,) = _broadcast_numeric_operands(x)
    signs = np.sign(values)
    signs = np.where(signs == 0.0, 1.0, signs)
    return _broadcast_result(
        reference,
        np.where(np.abs(values) < eps, signs * eps, values),
    )


def _safe_div_eps(x, y, eps):
    reference, (numerator, denominator) = _broadcast_numeric_operands(x, y)
    denominator = _floor_abs(denominator, eps)
    return _broadcast_result(reference, numerator / denominator)


def _max2(x, y):
    reference, (left, right) = _broadcast_numeric_operands(x, y)
    return _broadcast_result(reference, np.maximum(left, right))


def _min2(x, y):
    reference, (left, right) = _broadcast_numeric_operands(x, y)
    return _broadcast_result(reference, np.minimum(left, right))


def _comparison(x, y, comparator):
    reference, (left, right) = _broadcast_numeric_operands(x, y)
    unknown = np.isnan(left) | np.isnan(right)
    compared = comparator(left, right).astype(float)
    return _broadcast_result(reference, np.where(unknown, np.nan, compared))


def _gt(x, y):
    return _comparison(x, y, np.greater)


def _lt(x, y):
    return _comparison(x, y, np.less)


def _bool_to_float(condition):
    reference, (values,) = _broadcast_numeric_operands(condition)
    return _broadcast_result(
        reference,
        np.where(np.isfinite(values), np.where(values != 0.0, 1.0, 0.0), np.nan),
    )


def _where_gt(x, condition, threshold):
    reference, (values, condition_values) = _broadcast_numeric_operands(x, condition)
    out = np.where(condition_values > float(threshold), values, np.nan)
    return _broadcast_result(reference, out)


def _if_else(condition, x, y, threshold):
    reference, (condition_values, true_values, false_values) = (
        _broadcast_numeric_operands(condition, x, y)
    )
    out = np.where(
        condition_values > float(threshold), true_values, false_values
    )
    return _broadcast_result(
        reference,
        np.where(np.isfinite(condition_values), out, np.nan),
    )


def _cs_rank(x):
    counts = x.count(axis=1).astype(float)
    denom = counts.sub(1.0).replace(0.0, np.nan)
    out = x.rank(axis=1, method="average").sub(1.0).div(denom, axis=0)
    single = counts == 1.0
    if single.any():
        out.loc[single] = x.loc[single].notna().astype(float).replace(0.0, np.nan) * 0.5
    return out


def _cs_zscore(x):
    return x.sub(x.mean(axis=1), axis=0).div(x.std(axis=1, ddof=0).replace(0.0, np.nan), axis=0)


def _cs_demean(x): return x.sub(x.mean(axis=1), axis=0)
def _cs_scale(x): return x.div(x.abs().sum(axis=1).replace(0.0, np.nan), axis=0)


def _cs_quantile_bucket(x, bins):
    bins = int(bins)
    return np.floor(x.rank(axis=1, pct=True) * bins).clip(0, bins - 1).where(x.notna())


def _cs_rank_to_normal(x):
    pct = _cs_rank(x).clip(1e-4, 1.0 - 1e-4)
    return np.log(pct / (1.0 - pct))


def _cs_rank_where(x, condition, threshold):
    return _cs_rank(_where_gt(x, condition, threshold))


def _cs_quantile_bucket_where(x, condition, threshold, bins):
    return _cs_quantile_bucket(_where_gt(x, condition, threshold), bins)


def _ts_lag(x, window): return x.shift(int(window))
def _ts_delta(x, window): return x.diff(int(window))
def _ts_mean(x, window): return x.rolling(int(window), min_periods=int(window)).mean()
def _ts_std(x, window): return x.rolling(int(window), min_periods=int(window)).std(ddof=0)
def _ts_sum(x, window): return x.rolling(int(window), min_periods=int(window)).sum()
def _ts_cumsum(x, window): return x.rolling(int(window), min_periods=1).sum()
def _ts_min(x, window): return x.rolling(int(window), min_periods=int(window)).min()
def _ts_max(x, window): return x.rolling(int(window), min_periods=int(window)).max()
def _ts_argmax(x, window):
    window = int(window)
    return x.rolling(window, min_periods=window).apply(
        lambda values: float(np.nanargmax(values)) / max(1, window - 1), raw=True
    )
def _ts_argmax_index(x, window):
    window = int(window)
    return x.rolling(window, min_periods=window).apply(
        lambda values: float(np.nanargmax(values)) + 1.0, raw=True
    )


def _ts_argmin_index(x, window):
    window = int(window)
    return x.rolling(window, min_periods=window).apply(
        lambda values: float(np.nanargmin(values)) + 1.0, raw=True
    )


def _ts_median(x, window): return x.rolling(int(window), min_periods=int(window)).median()
def _ts_skew(x, window): return x.rolling(int(window), min_periods=int(window)).skew()
def _ts_kurt(x, window): return x.rolling(int(window), min_periods=int(window)).kurt()
def _ts_quantile(x, window, q): return x.rolling(int(window), min_periods=int(window)).quantile(float(q))
def _ts_rank(x, window):
    def last_rank(values):
        if np.isnan(values).any(): return np.nan
        if len(values) <= 1: return 0.5
        last = values[-1]
        rank = np.sum(values < last) + (np.sum(values == last) + 1.0) / 2.0
        return float((rank - 1.0) / (len(values) - 1.0))
    return x.rolling(int(window), min_periods=int(window)).apply(last_rank, raw=True)


def _ts_zscore(x, window):
    return (x - _ts_mean(x, window)) / _ts_std(x, window).replace(0.0, np.nan)


def _ts_decay_linear(x, window):
    window = int(window)
    weights = np.arange(1, window + 1, dtype=float)
    weights /= weights.sum()
    return x.rolling(window, min_periods=window).apply(lambda v: float(np.dot(v, weights)), raw=True)

def _ts_path_range(x, window):
    window = int(window)
    def path_range(values):
        if not np.isfinite(values).all(): return np.nan
        path = np.cumsum(values, dtype=float)
        return float(np.max(path) - np.min(path))
    return x.rolling(window, min_periods=window).apply(path_range, raw=True)


def _ts_corr(x, y, window):
    x, y = _align_pair(x, y)
    return x.rolling(int(window), min_periods=int(window)).corr(y)


def _ts_cov(x, y, window):
    x, y = _align_pair(x, y)
    return x.rolling(int(window), min_periods=int(window)).cov(y)


def _ts_beta(x, y, window):
    x, y = _align_pair(x, y)
    return _ts_cov(x, y, window) / y.rolling(int(window), min_periods=int(window)).var().replace(0.0, np.nan)


def _ts_resid(x, y, window):
    beta = _ts_beta(x, y, window)
    return x - (_ts_mean(x, window) - beta * _ts_mean(y, window)) - beta * y

def _rolling_time_regression(x, window, mode):
    window = int(window)
    out = pd.DataFrame(np.nan, index=x.index, columns=x.columns, dtype=float)
    if window < 2 or len(x.index) < window:
        return out
    finite = x.notna()
    safe = x.where(finite, 0.0)
    sums = safe.rolling(window, min_periods=window).sum()
    sums_sq = (safe * safe).rolling(window, min_periods=window).sum()
    positions = pd.Series(np.arange(len(x.index), dtype=float), index=x.index)
    weighted = safe.mul(positions, axis=0).rolling(window, min_periods=window).sum()
    starts = positions - float(window - 1)
    local_weighted = weighted - sums.mul(starts, axis=0)
    time_mean = float(window - 1) / 2.0
    time_ss = float(window * (window * window - 1)) / 12.0
    numerator = local_weighted - time_mean * sums
    slope = numerator / time_ss
    valid = finite.astype(float).rolling(window, min_periods=window).sum() == window
    if mode == "slope":
        return slope.where(valid)
    mean = sums / float(window)
    if mode == "resid":
        return (x - mean - slope * time_mean).where(valid)
    value_ss = (sums_sq - sums * sums / float(window)).clip(lower=0.0)
    return ((numerator * numerator) / (time_ss * value_ss)).where(
        valid & (value_ss > 0.0)
    )


def _ts_slope(x, window):
    return _rolling_time_regression(x, window, "slope")


def _ts_time_r2(x, window):
    return _rolling_time_regression(x, window, "r2")


def _ts_time_resid(x, window):
    return _rolling_time_regression(x, window, "resid")


def _ts_intercept(x, y, window):
    beta = _ts_beta(x, y, window)
    return _ts_mean(x, window) - beta * _ts_mean(y, window)


def _ts_r2(x, y, window): return _ts_corr(x, y, window) ** 2


def main(datasources, start_date, end_date):
    if not isinstance(datasources, dict) or "bar1m" not in datasources:
        raise ValueError("datasources must contain logical bar1m")
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    if start > end:
        raise ValueError("start_date must not exceed end_date")
    query_start = (start - pd.Timedelta(days=max(30, _LOOKBACK * 2 + 10))).strftime("%Y-%m-%d")
    query_end = end.strftime("%Y-%m-%d")
    daily_parts = []
    membership_parts = []
    for chunk_start, chunk_end in _date_chunks(query_start, query_end):
        # BigQuant applies the filter boundary to the intraday timestamp.  A
        # bare YYYY-MM-DD end therefore means midnight and excludes that
        # session's minute bars.  Bind the end to the end of the requested day
        # while leaving the daily universe filter unchanged.
        bar_chunk_end = chunk_end + " 23:59:59"
        raw_bars = _query(
            datasources["bar1m"], _BAR_MAP.values(), chunk_start, bar_chunk_end
        )
        if not raw_bars.empty:
            daily_parts.append(_daily_bars(raw_bars))
        raw_membership = _query(
            _UNIVERSE_TABLE, _UNIVERSE_MAP.values(), chunk_start, chunk_end
        )
        if not raw_membership.empty:
            membership_parts.append(raw_membership)
    if not daily_parts or not membership_parts:
        raise ValueError("requested interval produced no bar or universe data")
    daily = pd.concat(daily_parts, ignore_index=True)
    raw_membership = pd.concat(membership_parts, ignore_index=True)
    aligned = _align_universe(daily, raw_membership)
    fields = {
        name: aligned.pivot(index="date", columns="instrument", values=name).sort_index()
        for name in _FIELD_NAMES
    }
    factor = (_ts_slope(_safe_div(fields['close'],fields['low']),50)) * _SCORE_DIRECTION
    factor = factor.replace([np.inf, -np.inf], np.nan)
    factor = factor.loc[(factor.index >= start.normalize()) & (factor.index <= end.normalize())]
    result = factor.stack().dropna().rename("factor").reset_index()
    result.columns = ["date", "instrument", "factor"]
    result = result.sort_values(["date", "instrument"], kind="mergesort", ignore_index=True)
    if result.duplicated(["date", "instrument"]).any():
        raise ValueError("submission produced duplicate keys")
    if list(result.columns) != ["date", "instrument", "factor"]:
        raise AssertionError("submission output contract changed")
    return result


In [ ]:
# Official BigAlpha public-data self-test.  This runs the factor and evaluator;
# it does not upload code or create a competition submission.
from bigmodule import M
import hashlib
import json

_SELFTEST_DATASOURCES = {'bar1m': 'bigalpha_2026_stock_bar1m', 'financial': 'bigalpha_2026_financial'}
_FACTOR_POOL_TABLE = 'bigalpha_2026_factorlib'
_PLATFORM_MANAGED_EXPOSURE_TABLE = 'bigalpha_2026_exposure'
_SELFTEST_START_DATE = '2024-01-02'
_SELFTEST_END_DATE = '2024-12-31'
_SELFTEST_CANDIDATE_ID = 'BA_AUTO_02779408441947903086'
_EXPORT_MANIFEST_SHA256 = '3f47b589f899e7ac0321f01236b8f95c1bca0f2b427cb5ad6cd170d50b81c917'
_SUBMISSION_SOURCE_SHA256 = '9432992421695e51b6c34bfee8763d32ed1e179b3159372b0b4fd2a89c349a05'
_SCHEMA_PROFILE_SHA256 = 'e67060b5b1efe9a98e1370b9146e6a63e018f5c35c2f96ded46b1f98a834e394'


def _official_selftest_preflight(frame):
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("main() must return a pandas DataFrame")
    if list(frame.columns) != ["date", "instrument", "factor"]:
        raise ValueError("main() must return exactly date/instrument/factor")
    if frame.empty:
        raise ValueError("main() returned no factor rows")
    if frame.duplicated(["date", "instrument"]).any():
        raise ValueError("main() returned duplicate date/instrument keys")
    dates = pd.to_datetime(frame["date"], errors="coerce")
    factors = pd.to_numeric(frame["factor"], errors="coerce")
    if dates.isna().any():
        raise ValueError("main() returned invalid dates")
    if frame["instrument"].astype(str).str.strip().eq("").any():
        raise ValueError("main() returned empty instruments")
    if not np.isfinite(factors.to_numpy(dtype=float)).all():
        raise ValueError("main() returned non-finite factor values")
    normalized = pd.DataFrame(
        {
            "date": dates.astype("int64"),
            "instrument": frame["instrument"].astype(str),
            "factor": factors.astype("float64"),
        }
    )
    row_hashes = pd.util.hash_pandas_object(normalized, index=False)
    return {
        "rows": int(len(frame)),
        "dates": int(dates.dt.normalize().nunique()),
        "instruments": int(frame["instrument"].nunique()),
        "factor_sha256": hashlib.sha256(row_hashes.to_numpy().tobytes()).hexdigest(),
    }


def _official_selftest_score_summary(evaluation):
    try:
        factor_analyze = evaluation["factor_analyze"]
        factor_regression = evaluation["factor_regression"]
        per_factor_scores = factor_regression["per_factor_scores"]
    except (KeyError, TypeError) as exc:
        raise ValueError(
            "official evaluator output is missing factor score components"
        ) from exc
    if not isinstance(per_factor_scores, pd.DataFrame):
        raise TypeError("official evaluator per_factor_scores must be a DataFrame")
    required_columns = {
        "factor",
        "model_score",
        "abs_weight_mean",
        "abs_weight_std",
        "selection_rate",
    }
    missing = sorted(required_columns - set(per_factor_scores.columns))
    if missing:
        raise ValueError(
            "official evaluator per_factor_scores is missing columns: " + repr(missing)
        )
    candidate_rows = per_factor_scores.loc[
        per_factor_scores["factor"].astype(str).eq("factor")
    ]
    if len(candidate_rows) != 1:
        raise ValueError("official evaluator must return exactly one candidate score row")
    candidate_score = candidate_rows.iloc[0]
    numeric = {
        "ic_mean": factor_analyze["ic_mean"],
        "ic_ir": factor_analyze["ic_ir"],
        "sharpe_ratio": factor_analyze["sharpe_ratio"],
        "stress_ic_ir": factor_analyze["stress_ic_ir"],
        "model_score": candidate_score["model_score"],
        "abs_weight_mean": candidate_score["abs_weight_mean"],
        "abs_weight_std": candidate_score["abs_weight_std"],
        "selection_rate": candidate_score["selection_rate"],
    }
    summary = {}
    for name, value in numeric.items():
        parsed = float(value)
        if not np.isfinite(parsed):
            raise ValueError("official evaluator returned non-finite " + name)
        summary[name] = parsed
    return {
        "schema_id": "bigalpha_official_selftest_score_summary_v1",
        "competition_submission": False,
        "candidate_id": _SELFTEST_CANDIDATE_ID,
        "official_public_percentiles_reproducible": False,
        "factor_pool_feature_count": int(len(per_factor_scores) - 1),
        **summary,
    }


def run_official_selftest():
    factor_data = main(
        dict(_SELFTEST_DATASOURCES),
        _SELFTEST_START_DATE,
        _SELFTEST_END_DATE,
    )
    evidence = _official_selftest_preflight(factor_data)
    evidence.update(
        {
            "schema_id": "bigalpha_official_selftest_evidence_v1",
            "competition_submission": False,
            "official_data_only": True,
            "candidate_id": _SELFTEST_CANDIDATE_ID,
            "start_date": _SELFTEST_START_DATE,
            "end_date": _SELFTEST_END_DATE,
            "submission_source_sha256": _SUBMISSION_SOURCE_SHA256,
            "export_manifest_sha256": _EXPORT_MANIFEST_SHA256,
            "schema_profile_sha256": _SCHEMA_PROFILE_SHA256,
            "factor_pool_table": _FACTOR_POOL_TABLE,
            "platform_managed_exposure_table": _PLATFORM_MANAGED_EXPOSURE_TABLE,
            "official_evaluator": "M.bigalpha_eval._latest",
            "process_pools": False,
        }
    )
    print(json.dumps(evidence, ensure_ascii=False, sort_keys=True))
    factor_pool_result = dai.query(
        "SELECT * FROM " + _FACTOR_POOL_TABLE,
        filters={"date": [_SELFTEST_START_DATE, _SELFTEST_END_DATE]},
    )
    factor_pool = (
        factor_pool_result.df()
        if callable(getattr(factor_pool_result, "df", None))
        else factor_pool_result
    )
    if not isinstance(factor_pool, pd.DataFrame) or factor_pool.empty:
        raise ValueError("official factor pool query returned no DataFrame rows")
    evaluation = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )
    score_summary = _official_selftest_score_summary(evaluation)
    print(json.dumps(score_summary, ensure_ascii=False, sort_keys=True))
    print(
        json.dumps(
            {
                "schema_id": "bigalpha_official_selftest_completion_v1",
                "competition_submission": False,
                "candidate_id": _SELFTEST_CANDIDATE_ID,
                "evaluation_completed": True,
            },
            ensure_ascii=False,
            sort_keys=True,
        )
    )
    return {
        "factor_data": factor_data,
        "factor_pool": factor_pool,
        "evaluation": evaluation,
        "evidence": evidence,
        "score_summary": score_summary,
    }


if __name__ == "__main__":
    official_selftest_outputs = run_official_selftest()
